# Independent Ensemble Model Tester
This notebook isolates the anomaly detection ensemble component and runs it directly against the dataset to test its output.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

# Add the webapp folder to the path
sys.path.insert(0, os.path.join(os.getcwd(), 'webapp'))
from ensemble import CybSecEnsemble
from model_utils import ModelHandler, parse_logs

In [ ]:
print("Initializing ModelHandler and loading models...")
handler = ModelHandler()
print("Models loaded successfully!")

In [ ]:
log_file = 'synthetic_logs_10k.csv'
print(f"Loading {log_file}...")
df = parse_logs(log_file)
print(f"Loaded {len(df)} log lines.")
display(df.head())

In [ ]:
print("Running Autoencoder inference...")
X_ae = handler.preprocess_ae(df)
reconstructions = handler.ae_model.predict(X_ae, verbose=0)
mse_vals = np.mean(np.square(X_ae - reconstructions), axis=1)
print("Done!")

In [ ]:
print("Running Isolation Forest inference...")
X_iso = handler.preprocess_iso(df)
iso_scores_all = handler.iso_forest.decision_function(X_iso)
print("Done!")

In [ ]:
print("Applying Ensemble weights (55% Autoencoder, 45% Isolation Forest)...")
is_anomaly, ensemble_scores = handler.ensemble_model.predict(mse_vals, iso_scores_all)

anomalies_df = df[is_anomaly].copy()
anomalies_df['ensemble_error'] = ensemble_scores[is_anomaly]

print(f"\nDetected {len(anomalies_df)} anomalies with the ensemble model.\n")

In [ ]:
print("Top 10 highest-severity ensemble anomalies:\n")
top_anomalies = anomalies_df.sort_values(by='ensemble_error', ascending=False).head(10)
display(top_anomalies)